# 10장 — 되풀이되는 이야기: 시스템 원형

웹 챕터(10장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다:
여섯 원형(성장의 한계·문제 전가·실패하는 처방·목표 침식·공유지의 비극·부익부)의 작은 시뮬레이션과 정책 비교.
모두 오일러법(DT = 1/16개월), 웹 sim.ts 의 arch* 함수와 같은 식이에요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch10')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

In [ ]:
DT = 1 / 16

def sim(deriv, s0, T, clamp0=False):
    # 웹 archSim 과 같은 계산: 상태 목록의 시간 기록을 돌려준다
    s, out = list(s0), [list(s0)]
    for i in range(round(T / DT)):
        d = deriv(s, i * DT)
        s = [a + b * DT for a, b in zip(s, d)]
        if clamp0:
            s = [max(0.0, v) for v in s]
        out.append(list(s))
    return out

at = lambda path, month, k=0: path[round(month / DT)][k]
print('엔진 준비 완료')

## ① 성장의 한계 (Limits to Growth)

로지스틱 $dS/dt = rS(1 - S/K)$, 처음 10명. 성장 엔진을 두 배로(r 0.5 → 1.0) 돌리는 것과 한계를 넓히는 것(K 1,000 → 1,500)을 비교합니다.

In [ ]:
def limits(r, K):
    return sim(lambda s, t: [r * s[0] * (1 - s[0] / K)], [10], 36)

base, engine, wider = limits(0.5, 1000), limits(1.0, 1000), limits(0.5, 1500)
for name, p in [('기본', base), ('엔진 강화(r 1.0)', engine), ('한계 넓히기(K 1,500)', wider)]:
    print(f'{name:14s}  12개월 {at(p, 12):7.1f}명   36개월 {at(p, 36):7.1f}명')
assert round(at(base, 36)) == round(at(engine, 36)) == 1000     # 엔진을 밀어도 같은 천장
assert round(at(wider, 36)) == 1500
print('✅ 성장 엔진을 두 배로 돌려도 천장은 1,000명 그대로, 한계를 넓혀야 1,500명')

## ② 문제 전가 (Shifting the Burden)

증상 S, 역량 C. $dS/dt = 20(1 - C) - (\text{빠른}\cdot 0.5 + 1/6)\,S$, $dC/dt = \text{근본}\cdot 0.15(1 - C) - \text{빠른}\cdot 0.05\,C$. 처음 S = 30, C = 0.5.

In [ ]:
def burden(uq, uf):
    return sim(lambda s, t: [20 * (1 - s[1]) - (uq * 0.5 + 1 / 6) * s[0],
                             uf * 0.15 * (1 - s[1]) - uq * 0.05 * s[1]], [30, 0.5], 36)

for name, (uq, uf) in [('빠른 처방만', (1, 0)), ('근본 처방만', (0, 1)), ('둘 다', (1, 1))]:
    p = burden(uq, uf)
    S = [x[0] for x in p]
    print(f'{name:8s}  증상 3개월 {at(p, 3):5.1f}  12개월 {at(p, 12):5.1f}  36개월 {at(p, 36):5.1f}  (최고 {max(S):5.1f}, 최저 {min(S):5.1f})  역량 36개월 {at(p, 36, 1):.2f}')
q, f, b = burden(1, 0), burden(0, 1), burden(1, 1)
assert round(at(q, 3)) == 18 and round(at(q, 36), 1) == 27.3 and round(at(q, 36, 1), 2) == 0.08
assert round(max(x[0] for x in f)) == 37 and round(at(f, 36), 1) == 1.3
assert round(at(b, 36), 1) == 7.5
print('✅ 빠른 처방만: 30 → 18 → 27(역량 붕괴) · 근본만: 37까지 나빠 보이다 1.3 · 둘 다: 7.5')

## ③ 실패하는 처방 (Fixes that Fail)

$dP/dt = 10 - P/4 - \text{처방} + 1.3 \times (\text{6개월 전 처방})$, 처방 $= 0.5P$, 처음 P = 40. 6개월 전 처방은 따로 기록해 두고 꺼내 씁니다.

In [ ]:
def fixes(k, T=48, D=6):
    lag = round(D / DT)
    P, hist, out = 40.0, [], [40.0]
    for i in range(round(T / DT)):
        fix = k * P
        past = hist[i - lag] if i >= lag else 0.0
        hist.append(fix)
        P += (10 - P / 4 - fix + 1.3 * past) * DT
        out.append(P)
    return out

none, fixed = fixes(0), fixes(0.5)
pick = lambda path, m: path[round(m / DT)]
for m in [0, 2, 6, 12, 24, 48]:
    print(f'{m:2d}개월  처방 안 함 {pick(none, m):5.1f}건   처방 {pick(fixed, m):5.1f}건')
assert all(abs(v - 40) < 1e-9 for v in none)
assert round(pick(fixed, 6), 1) == 13.6 and round(pick(fixed, 24)) == 45 and round(pick(fixed, 48)) == 65
print('✅ 처방: 6개월 13.6건 → 2년 45건 → 4년 65건 (처방 안 할 때 40건보다 나쁨)')

## ④ 목표 침식 (Eroding Goals)

품질 P, 목표 G. $dP/dt = 0.2(G - P) - 0.02P$, 목표 침식이면 $dG/dt = -(G - P)/6$. 처음 P = 80, G = 95.

In [ ]:
def goals(erode):
    return sim(lambda s, t: [0.2 * (s[1] - s[0]) - 0.02 * s[0], -(s[1] - s[0]) / 6 if erode else 0.0], [80, 95], 48)

fixed_g, eroded = goals(False), goals(True)
for m in [12, 24, 48]:
    print(f'{m:2d}개월  목표 고정: 품질 {at(fixed_g, m):5.1f}   목표 침식: 품질 {at(eroded, m):5.1f}, 목표 {at(eroded, m, 1):5.1f}')
assert round(at(fixed_g, 48)) == 86 and round(at(eroded, 48)) == 56
print('✅ 목표를 지키면 86%, 목표를 내리기 시작하면 4년 뒤 56%')

## ⑤ 공유지의 비극 (Tragedy of the Commons)

공용 자원 R은 $0.3R(1 - R/100)$ 만큼 다시 차오르고, 두 팀의 사용 욕구 A·B는 매달 8%씩 커집니다.
실제 사용량 $= (A + B)\,R/(R + 5)$. 규칙: 팀마다 3.75 상한(지속 가능한 총량 7.5). 처음 R = 80, A = B = 2. 자원은 0 아래로 내려가지 않게 묶습니다.

In [ ]:
def commons(rule):
    def d(s, t):
        R, A, B = s
        H = (A + B) * R / (R + 5)
        dA = 0.0 if rule and A >= 3.75 else 0.08 * A
        dB = 0.0 if rule and B >= 3.75 else 0.08 * B
        return [0.3 * R * (1 - R / 100) - H, dA, dB]
    p = sim(d, [80, 2, 2], 48, clamp0=True)
    used = [(x[1] + x[2]) * x[0] / (x[0] + 5) for x in p]
    return p, used

(pf, uf_), (pr, ur) = commons(False), commons(True)
gone = next(i * DT for i, x in enumerate(pf) if x[0] < 0.01)
print(f'각자 늘림: 사용량 최고 {max(uf_):.1f}, 자원이 바닥난 시점 {gone:.1f}개월, 48개월 사용량 {uf_[-1]:.2f}')
print(f'총량 규칙: 48개월 사용량 {ur[-1]:.2f}, 자원 {pr[-1][0]:.1f}')
assert round(max(uf_), 1) == 15.7 and gone < 24 and uf_[-1] < 0.01
assert round(ur[-1]) == 7 and round(pr[-1][0]) == 63
print('✅ 각자 늘리면 한때 15.7까지 쓰다 2년 안에 바닥 · 규칙을 두면 약 7을 계속')

## ⑥ 부익부 (Success to the Successful)

예산 100을 성과의 제곱 비율로 나누고, 성과는 받은 예산을 3개월에 걸쳐 따라갑니다. 52 : 48 출발. 최소 30% 보장과 비교.

In [ ]:
def success(floor):
    def d(s, t):
        A, B = s
        sa = A * A / (A * A + B * B)
        if floor:
            sa = min(max(sa, 0.3), 0.7)
        return [(100 * sa - A) / 3, (100 * (1 - sa) - B) / 3]
    return sim(d, [52, 48], 36)

pure, floored = success(False), success(True)
for m in [6, 12, 36]:
    print(f'{m:2d}개월  몰아주기 A {at(pure, m):5.1f} : B {at(pure, m, 1):5.1f}    30% 보장 A {at(floored, m):5.1f} : B {at(floored, m, 1):5.1f}')
assert round(at(pure, 12)) == 90 and round(at(pure, 12, 1)) == 10
assert round(at(pure, 36)) == 100 and round(at(floored, 36, 1)) == 30
print('✅ 52:48 → 1년 90:10 → 3년 100:0 · 최소 30% 보장이면 70:30')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import numpy as np
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
ts36 = np.arange(0, 36 + 1e-9, DT); ts48 = np.arange(0, 48 + 1e-9, DT)
ax = axes[0][0]
for p, lab in [(base, '기본'), (engine, '엔진 강화'), (wider, '한계 넓히기')]:
    ax.plot(ts36, [x[0] for x in p], label=lab)
ax.set_title('① 성장의 한계'); ax.legend(fontsize=8)
ax = axes[0][1]
for p, lab in [(q, '빠른 처방만'), (f, '근본 처방만'), (b, '둘 다')]:
    ax.plot(ts36, [x[0] for x in p], label=lab)
ax.set_title('② 문제 전가 (증상)'); ax.legend(fontsize=8)
ax = axes[0][2]
ax.plot(ts48, none, label='처방 안 함'); ax.plot(ts48, fixed, label='처방')
ax.set_title('③ 실패하는 처방 (문제)'); ax.legend(fontsize=8)
ax = axes[1][0]
ax.plot(ts48, [x[0] for x in fixed_g], label='품질(목표 고정)')
ax.plot(ts48, [x[0] for x in eroded], label='품질(목표 침식)')
ax.plot(ts48, [x[1] for x in eroded], '--', label='목표(침식)')
ax.set_title('④ 목표 침식'); ax.legend(fontsize=8)
ax = axes[1][1]
ax.plot(ts48, uf_, label='각자 늘림'); ax.plot(ts48, ur, label='총량 규칙')
ax.set_title('⑤ 공유지의 비극 (실제 사용량)'); ax.legend(fontsize=8)
ax = axes[1][2]
ax.plot(ts36, [x[0] for x in pure], label='A (몰아주기)'); ax.plot(ts36, [x[1] for x in pure], label='B (몰아주기)')
ax.plot(ts36, [x[1] for x in floored], '--', label='B (30% 보장)')
ax.set_title('⑥ 부익부 (성과)'); ax.legend(fontsize=8)
for row in axes:
    for ax in row:
        ax.set_xlabel('개월'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 오늘의 발견

- 여섯 원형 모두 **단기에 좋아 보이는 길과 장기에 좋은 길이 다르다**. 사이에 대개 지연이 있다.
- 성장의 한계: 엔진을 밀어도 천장은 그대로(1,000명) → 한계를 넓혀라(1,500명).
- 문제 전가: 빠른 처방만이면 30 → 18 → 27(역량 0.08). 근본 처방과 함께 써라(7.5).
- 실패하는 처방: 13.6건까지 줄었다가 4년 뒤 65건(처방 안 할 때 40건). 지연된 부작용을 계산에 넣어라.
- 목표 침식: 목표를 지키면 86%, 내리기 시작하면 56%. 목표를 바깥 기준에 묶어라.
- 공유지의 비극: 각자 늘리면 2년 안에 고갈. 총량 규칙이면 약 7을 계속.
- 부익부: 52:48 → 100:0. 기본 몫을 보장하면 70:30.
- **레버리지**: 숫자보다 구조(고리·지연·정보·규칙·목표)를 바꿔라.

다음 장(11장 AI가 만드는 고리)에서 모델 붕괴·보상 해킹·플라이휠을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**